## Import Packages

In [2]:
import numpy as np
import pandas as pd

import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

/opt/anaconda3/envs/tf-clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
input_vocab = {'<SOS>': 0,
               'lets':1,
               'to': 2,
               'go':3}

output_vocab = {'<SOS>': 0,
                'ir': 1,
                'vamos': 2,
                'y': 3,
                '<EOS>': 4}

inputs = torch.tensor([[1,3],
                       [2,3]])

labels = torch.tensor([[2],
                       [1]])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Position Encoding

In [4]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model = 2, max_len=1):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)
        div_term = 1 / 10000 ** (torch.arange(start=0, end=d_model, step=2).float() / d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

## Attention

In [5]:
class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = 0
        self.col_dim = 1

    def forward(self, encoding_for_q, encoding_for_k, encoding_for_v, mask=None):
        q = self.W_q(encoding_for_q)
        k = self.W_k(encoding_for_k)
        v = self.W_v(encoding_for_v)

        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims / (q.size(self.col_dim) ** 0.5)

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask = mask, value = -1.e9)

        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)

        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

## Encoder
- Word Embedding
- Position Encoding
- Self-Attention
- Residual Connections

In [12]:
class Encoder(nn.Module):
    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        super().__init__()
        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        self.pe = PositionEncoding(d_model = d_model,
                                   max_len = max_len)
        
        self.self_attention = Attention(d_model = d_model)

    def forward(self, token_ids):
        word_embeddings = self.we(token_ids)
        position_encoded = self.pe(word_embeddings)
        self_attention_values = self.self_attention(position_encoded,
                                                         position_encoded,
                                                         position_encoded)
        
        output_values = position_encoded + self_attention_values

        return output_values

## Decoder
- Word Embedding
- Position Encoding
- Self-Attention
- Residual Connections

- Encoder-Decoder Attention
- A full connected layer
- SoftMax

In [ ]:
class Decoder(nn.Module):
    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        super().__init__()
        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        self.pe = PositionEncoding(d_model=d_model,
                                   max_len=max_len)
        
        self.self_attention = Attention(d_model=d_model)
        
        self.enc_dec_attention = Attention(d_model=d_model)

        self.fc_layer = nn.Linear(in_features=d_model, 
                                  out_features=num_tokens)
        
        self.row_dim = 0
        self.col_dim = 1

    def forward(self, token_ids, encoder_values):
        word_embeddings = self.we(token_ids)
        position_encoded = self.pe(word_embeddings)

        mask = torch.tril(torch.ones((token_ids.size(dim=self.row_dim), token_ids.size(dim=self.row_dim)), device=token_ids.device))

        mask = mask == 0

        self.self_attention_values = self.self_attention(position_encoded,
                                                         position_encoded,
                                                         position_encoded,
                                                         mask = mask)
        residual_connection_values = position_encoded + self.self_attention_values
        enc_dec_attention_values = self.enc_dec_attention(residual_connection_values,
                                                               encoder_values,
                                                               encoder_values)
        residual_connection_values = enc_dec_attention_values + residual_connection_values
        fc_layer_output = self.fc_layer(residual_connection_values)

        return fc_layer_output

## Transformer

In [90]:
class Transformer(L.LightningModule):
    def __init__(self, input_size, output_size, d_model=2, max_len=3):
        super().__init__()

        self.encoder = Encoder(num_tokens=len(input_vocab), d_model = d_model, max_len = max_len)
        self.decoder = Decoder(num_tokens=len(output_vocab), d_model=d_model, max_len=max_len)

        self.loss = nn.CrossEntropyLoss()
    
    def forward(self, inputs, labels):
        encoder_values = self.encoder(inputs)
        output_presoftmax = self.decoder(labels, encoder_values)

        return(output_presoftmax)
    
    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)
    
    def training_step(self, batch, batch_idx): 
        
        input_i, label_i = batch # collect input
        
        ## First, let's append the <SOS> token to tokens used as input to the Encoder...
        input_tokens = torch.cat((torch.tensor([0], device=input_i.device, dtype=input_i.dtype), input_i[0]))

        
        ## ...and to the tokens used as input to the decoder.
        teacher_forcing = torch.cat((torch.tensor([0], device=label_i.device, dtype=label_i.dtype), label_i[0]))
        
        ## Now let's add the <EOS> token to the end of the known output
        expected_output = torch.cat((label_i[0], torch.tensor([4], device=label_i.device, dtype=label_i.dtype)))

        output_i = self.forward(input_tokens, teacher_forcing)
        loss = self.loss(output_i, expected_output)

        return loss

In [109]:
transformer = Transformer(len(input_vocab), len(output_vocab), d_model=2, max_len=3)

Seed set to 42
Seed set to 42


In [110]:
trainer = L.Trainer(max_epochs=20)
trainer.fit(transformer, train_dataloaders=dataloader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder │ Encoder          │     20 │ train │     0 │
│ 1 │ decoder │ Decoder          │     49 │ train │     0 │
│ 2 │ loss    │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 69                                                                                               
Non-trainable params: 0                                                                                            
Total params: 69                                                                                                   
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 20                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=20` reached.


In [111]:
## First, a reminder of our input and output vocabularies...
# input_vocab = {'<SOS>': 0, # Start
#                'lets': 1,
#                'to': 2,
#                'go': 3}

# output_vocab = {'<SOS>': 0, # Start
#                 'ir': 1,
#                 'vamos': 2,
#                 'y': 3,
#                 '<EOS>': 4} # End

max_length = 3
row_dim = 0
col_dim = 1

## Encode the user input...
encoder_values = transformer.encoder(torch.tensor([0, 1, 3])) # <SOS> let's go # Expecting: 0, 2, 4 = <SOS> vamos <EOS>
# encoder_values = transformer.encoder(torch.tensor([0, 2, 3])) # <SOS> to go  # Expecting: 0, 1, 4 = <SOS> ir <EOS>
    
## Since we initialize the decoder with the <SOS> token, we
## can consider that <SOS> to be the first predicted token
predicted_ids = torch.tensor([0]) # set the first predicted token to <SOS> to initialize the decoder
for i in range(max_length):
    ## given the current predicted tokens and the encoded input, 
    ## predict the next token with the decoder
    ## NOTE: "prediction" is the output from the fully connected layer,
    ##      not a softmax() function. We could, if we wanted to,
    ##      Run "prediction" through a softmax() function, but 
    ##      since we're going to select the item with the largest value
    ##      we can just use argmax instead...
    prediction = transformer.decoder(predicted_ids, encoder_values)

    ## Use argmax() to select the id of the predicted token
    predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])
    ## add the predicted token id to the list of predicted ids.
    predicted_ids = torch.cat((predicted_ids, predicted_id))
        
    if (predicted_id == 4): # if the prediction is <EOS>, then we are done
        break
        
print("\npredicted_ids:", predicted_ids)


predicted_ids: tensor([0, 2, 4])


In [122]:
transformer.encoder.we.weight.data.round(decimals=2)

tensor([[ 1.3900, -0.0000],
        [-0.0600, -0.0400],
        [-2.6700,  1.1600],
        [ 3.5400, -0.7400]])

In [124]:
we = nn.Embedding(num_embeddings=4,
             embedding_dim=2)

In [126]:
we(inputs)

tensor([[[ 0.6488,  0.4496],
         [ 0.0274, -0.7916]],

        [[ 0.3220, -1.0503],
         [ 0.0274, -0.7916]]], grad_fn=<EmbeddingBackward0>)

In [112]:
max_length = 3

In [113]:
input_vocab

{'<SOS>': 0, 'lets': 1, 'to': 2, 'go': 3}

In [114]:
output_vocab

{'<SOS>': 0, 'ir': 1, 'vamos': 2, 'y': 3, '<EOS>': 4}

In [115]:
encoder_values = transformer.encoder(torch.tensor([0,1,3]))

In [116]:
predicted_ids = torch.tensor([0])
print("predicted_ids: ", predicted_ids)

prediction = transformer.decoder(predicted_ids, encoder_values)
print('prediction: ', prediction)


predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])
print('prediction_id: ', predicted_id)

predicted_ids = torch.cat((predicted_ids, predicted_id))
print('predicted_ids: ', predicted_ids)

predicted_ids:  tensor([0])
prediction:  tensor([[-10.9095,  -1.1789,   6.8848, -10.8349,  -2.3823]],
       grad_fn=<AddmmBackward0>)
prediction_id:  tensor([2])
predicted_ids:  tensor([0, 2])


In [117]:
print("predicted_ids: ", predicted_ids)

prediction = transformer.decoder(predicted_ids, encoder_values)
print('prediction: ', prediction)

predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])
print('prediction_id: ', predicted_id)

predicted_ids = torch.cat((predicted_ids, predicted_id))
print('predicted_ids: ', predicted_ids)

predicted_ids:  tensor([0, 2])
prediction:  tensor([[-10.9095,  -1.1789,   6.8848, -10.8349,  -2.3823],
        [ -2.8181, -13.3719,  -2.5994,  -5.2065,  21.4904]],
       grad_fn=<AddmmBackward0>)
prediction_id:  tensor([4])
predicted_ids:  tensor([0, 2, 4])


In [118]:
print("predicted_ids: ", predicted_ids)

prediction = transformer.decoder(predicted_ids, encoder_values)
print('prediction: ', prediction)

predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])
print('prediction_id: ', predicted_id)

predicted_ids = torch.cat((predicted_ids, predicted_id))
print('predicted_ids: ', predicted_ids)

predicted_ids:  tensor([0, 2, 4])
prediction:  tensor([[-10.9095,  -1.1789,   6.8848, -10.8349,  -2.3823],
        [ -2.8181, -13.3719,  -2.5994,  -5.2065,  21.4904],
        [ -8.5105,  -1.7173,   5.1421,  -8.6002,  -0.1809]],
       grad_fn=<AddmmBackward0>)
prediction_id:  tensor([2])
predicted_ids:  tensor([0, 2, 4, 2])


In [119]:
print("predicted_ids: ", predicted_ids)

prediction = transformer.decoder(predicted_ids, encoder_values)
print('prediction: ', prediction)

predicted_id = torch.tensor([torch.argmax(prediction[-1,:])])
print('prediction_id: ', predicted_id)

predicted_ids = torch.cat((predicted_ids, predicted_id))
print('predicted_ids: ', predicted_ids)

predicted_ids:  tensor([0, 2, 4, 2])


RuntimeError: The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 0

#### Word-Embedding Process

In [219]:
input_words = {'<SOS>': 0,
            'I': 1,
            'Love': 2,
            'Machine': 3,
            'Learning': 4,
            'And': 5,
            'Math': 6}

output_words = {'<SOS>': 0,
                '我': 1,
                '爱': 2,
                '机': 3,
                '器': 4,
                '学': 5, 
                '习': 6,
                '和': 7,
                '数': 8,
                '学': 9,
                '<EOS>':10}


inputs = torch.tensor([0, 1, 2, 6])
labels = torch.tensor([0, 1, 2, 8, 9, 10])


we = nn.Embedding(num_embeddings=20,
             embedding_dim=2)
print(we(inputs))


de = nn.Embedding(num_embeddings=20,
             embedding_dim=2)
print(de(labels))

tensor([[-2.3692, -0.2767],
        [-0.3386, -0.8810],
        [ 0.3484,  0.6930],
        [-0.8164, -1.6489]], grad_fn=<EmbeddingBackward0>)
tensor([[-0.1440,  0.0518],
        [-0.3285, -2.2472],
        [-0.4479,  0.4235],
        [-0.2455,  0.2465],
        [ 0.1329,  0.1219],
        [ 0.4781,  0.2761]], grad_fn=<EmbeddingBackward0>)


#### Encoder - Position Encoding

In [210]:
max_len = 4
d_model = 2

position = torch.arange(start=0, end = max_len, step=1).float().unsqueeze(1)
print(position)

pe = torch.zeros(max_len, d_model)
print(pe)

div_dim = 1 / torch.tensor(10000.00) ** (torch.arange(start=0, end=d_model, step=2).float() / d_model)
div_dim

pe[:, 0::2] = torch.sin(position * div_dim)
print(pe)

pe[:, 1::2] = torch.cos(position * div_dim)
print(pe)

tensor([[0.],
        [1.],
        [2.],
        [3.]])
tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]])
tensor([[0.0000, 0.0000],
        [0.8415, 0.0000],
        [0.9093, 0.0000],
        [0.1411, 0.0000]])
tensor([[ 0.0000,  1.0000],
        [ 0.8415,  0.5403],
        [ 0.9093, -0.4161],
        [ 0.1411, -0.9900]])


In [215]:
Encoded_Values = we(inputs) + pe
Encoded_Values

tensor([[-0.7815, -0.7899],
        [ 0.5799,  0.1001],
        [-0.2041,  0.1097],
        [ 0.7679, -1.4876]], grad_fn=<AddBackward0>)

# Encoder

#### Encoder - Self-Attention Process

In [240]:
Encoded_Values = torch.tensor([[1.16, 0.23],
                  [0.57, 1.36],
                  [4.42, -2.16]])

Query_weights = torch.tensor([[2.22, 0.41],
               [0.17, -0.51]])

Key_weights = torch.tensor([[-1.82, 0.57],
               [1.36,-0.38]])

Value_weights = torch.tensor([[-0.43, -0.59],
               [1.33, -2.15]])

In [261]:
max_len = 20

In [262]:
e_d_model = Encoded_Values.size(1)
e_d_model

2

In [241]:
Q = Encoded_Values @ Query_weights
K = Encoded_Values @ Key_weights
V = Encoded_Values @ Value_weights

In [242]:
Q

tensor([[ 2.6143,  0.3583],
        [ 1.4966, -0.4599],
        [ 9.4452,  2.9138]])

In [243]:
K

tensor([[ -1.7984,   0.5738],
        [  0.8122,  -0.1919],
        [-10.9820,   3.3402]])

In [244]:
V

tensor([[-0.1929, -1.1789],
        [ 1.5637, -3.2603],
        [-4.7734,  2.0362]])

In [245]:
Q @ K.T

tensor([[ -4.4960,   2.0546, -27.5134],
        [ -2.9554,   1.3038, -17.9718],
        [-15.3143,   7.1122, -93.9945]])

In [247]:
sim = Q @ K.T / (torch.tensor(e_d_model) ** 0.5)
sim

tensor([[ -3.1791,   1.4528, -19.4549],
        [ -2.0898,   0.9219, -12.7080],
        [-10.8289,   5.0291, -66.4642]])

In [248]:
sm = F.softmax(sim, dim =1).data.round(decimals=2)
sm

tensor([[0.0100, 0.9900, 0.0000],
        [0.0500, 0.9500, 0.0000],
        [0.0000, 1.0000, 0.0000]])

In [249]:
self_attention = sm @ V
self_attention

tensor([[ 1.5461, -3.2395],
        [ 1.4759, -3.1562],
        [ 1.5637, -3.2603]])

In [250]:
encoder = self_attention + Encoded_Values
encoder

tensor([[ 2.7061, -3.0095],
        [ 2.0459, -1.7962],
        [ 5.9837, -5.4203]])

# Decoder

### Decoder - Self-Attention

In [298]:
Decoder_Values = torch.tensor([[-2.53, 0.03],
                              [1.55, 1.27]])

D_Query_Weights = torch.tensor([[-0.19, 0.24],
                                [0.64, 1.47]])

D_Key_Weights = torch.tensor([[-0.08, 0.38],
                                [1.18, 0.67]])

D_Value_Weights = torch.tensor([[1.26, 1.10],
                                [-0.71, 0.05]])

In [299]:
D_Q = Decoder_Values @ D_Query_Weights
D_Q.round(decimals=2)

tensor([[ 0.5000, -0.5600],
        [ 0.5200,  2.2400]])

In [300]:
D_K = Decoder_Values @ D_Key_Weights
D_K.round(decimals=2)

tensor([[ 0.2400, -0.9400],
        [ 1.3700,  1.4400]])

In [301]:
D_V = Decoder_Values @ D_Value_Weights
D_V.round(decimals=2)

tensor([[-3.2100, -2.7800],
        [ 1.0500,  1.7700]])

In [302]:
D_sims = D_Q @ D_K.T / torch.sqrt(torch.tensor(e_d_model))
D_sims

tensor([[ 0.4589, -0.0874],
        [-1.4031,  2.7833]])

### Decoder - Mask

In [303]:
mask = torch.tril(torch.ones(D_sims.size(0), e_d_model))
print(mask)

mask = mask == 0
print(mask)

tensor([[1., 0.],
        [1., 1.]])
tensor([[False,  True],
        [False, False]])


In [304]:
D_sims = D_sims.masked_fill(mask=mask, value=-1e9)
D_sims

tensor([[ 4.5886e-01, -1.0000e+09],
        [-1.4031e+00,  2.7833e+00]])

In [305]:
d_attention_scores = F.softmax(D_sims, dim = 1) @ D_V
decoder = d_attention_scores + Decoder_Values
decoder

tensor([[-5.7391, -2.7515],
        [ 2.5375,  2.9704]])

# Encoder - Decoder Attention

In [306]:
encoder

tensor([[ 2.7061, -3.0095],
        [ 2.0459, -1.7962],
        [ 5.9837, -5.4203]])

In [307]:
decoder

tensor([[-5.7391, -2.7515],
        [ 2.5375,  2.9704]])

In [308]:
ED_Query_Weights = torch.tensor([[0.9, 1.32],
                                [1.00,0.38]])

ED_Key_Weights = torch.tensor([[0.94,1.28],
                               [-0.70,-0.97]])

ED_Value_Weights = torch.tensor([[-1.03,1.73],
                                 [1.11, -1.49]])

In [309]:
ED_Q = decoder @ ED_Query_Weights
ED_K = encoder @ ED_Key_Weights
ED_V = encoder @ ED_Value_Weights

In [310]:
ED_Q

tensor([[-7.9167, -8.6212],
        [ 5.2541,  4.4783]])

In [311]:
ED_K

tensor([[ 4.6504,  6.3831],
        [ 3.1805,  4.3611],
        [ 9.4189, 12.9168]])

In [312]:
ED_V

tensor([[ -6.1278,   9.1657],
        [ -4.1011,   6.2157],
        [-12.1797,  18.4281]])

In [323]:
ED_sims = ED_Q @ ED_K.T / (ED_K.size(1) ** 0.5)
ED_sims

tensor([[ -64.9444,  -44.3896, -131.4686],
        [  37.4899,   25.6259,   75.8958]])

In [326]:
ED_Attention = F.softmax(ED_sims, dim=1) @ ED_V
ED_Attention

tensor([[ -4.1011,   6.2157],
        [-12.1797,  18.4281]])

In [327]:
ED_Attention + decoder

tensor([[-9.8402,  3.4642],
        [-9.6422, 21.3984]])

2